一个完整的 Agent 至少要包含俩个关键的部分：

- 模型：是 Agent 的大脑，负责推理、分析，规划任务步骤
- 工具：是 Agent 的手脚，负责执行模型规划的任务步骤，完成具体的操作

![image-20260415203111615](https://cdn.jsdelivr.net/gh/XVSHIFU/Picture-bed@img/image-20260415203111615.png)


# 基本用法

我们先通过一个案例快速回顾Agent定义的步骤，以及Agent的工作原理。

定义一个带有工具的Agent分为两步：

- 定义工具
- 定义Agent，绑定工具

首先，试用 tool 装饰器定义工具：

In [ ]:
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """获取指定位置的天气信息"""
    return f"Current weather in {location} is sunny"

接着，定义 Agent ，绑定工具：

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 2.创建智能体，并绑定工具
agent = create_agent(
    model="deepseek-chat",
    tools=[get_weather]
)

# 3.调用Agent
response = agent.invoke(
    {"messages": [HumanMessage(content="杭州今天天气如何?")]},
)

for message in response['messages']:
    message.pretty_print()

流程图：
![image-20260414214233705](https://cdn.jsdelivr.net/gh/XVSHIFU/Picture-bed@img/image-20260414214233705.png)

由此可见，所谓的工具，本质就是一个可调用的函数，要想让Agent知道有哪些工具可调用，该如何调用这些工具，就必须把这个函数的详细信息发送给模型。包括：
- 函数名
- 函数的作用
- 函数的参数和返回值信息

所以，定义工具的时候，关键就是把这些信息描述清楚即可。

# 自定义工具
在LangChain中，定义工具的过程被大大简化，与定义普通函数几乎没什么差别，只是在一些细节上需要注意。

首先，定义工具需要在函数上添加@tool装饰器。

例如，我们定义一个计算平方根的数学工具：

In [ ]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """计算一个数的平方根"""
    return x ** 0.5


智能体在工作时，需要将函数的名称、输入、作用传递给大模型，默认情况下这些信息的来源是：

- 工具名称：函数名
- 工具输入：函数入参
- 工具作用：函数的注释

当然，我们可以通过tool装饰器来覆盖上述信息：

## 通过装饰器定义工具名称

In [ ]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

## 通过装饰器定义工具作用描述

In [ ]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

## 通过装饰器定义工具入参约束

如果要覆盖工具的入参信息则会复杂很多，我们要借助于Pydantic或JSON约束。

例如，我们需要定义个查询天气的tool，借助于Pydantic来约束入参。
我们定义一个入参的模型，在模型中添加入参描述信息：

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

定义工具，使用定义的模型来约束入参：

In [ ]:
# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

工具定义好之后，调用方式与普通函数类似：

In [ ]:
# 调用数学工具
tool1.invoke({"x": 467})

# 调用查询天气工具
get_weather.invoke({"location": "杭州", "include_forecast": True})


注意：

在LangChain中，作为工具的函数有两个保留的参数名，你的自定义参数不能与之重复！他们是：

- config：用来传递运行时配置
- runtime：用来传递运行时上下文

具体可参考官方文档：[保留参数名（Reserved argument names）](https://docs.langchain.com/oss/python/langchain/tools#reserved-argument-names)，后续我们会讲到这两个参数的用法。

当创建智能体时，可以把定义好的工具传递给智能体，智能体就会知道有哪些工具可用，以及如何调用这些工具。

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model = "deepseek-chat",
    tools = [tool1, get_weather],
    system_prompt = "你是一个热心的AI智能助手，你将试用工具来解决用户的问题。"
)

接下来，调用智能体，向其提问，模型会自动根据用户问题判断：

- 是否需要调用工具？
- 该调用哪个工具？
- 该传递那些参数？

并且在调用工具之后，根据工具执行结果给用户生成响应。

In [ ]:
from langchain_core.messages import HumanMessage

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="467的平方根是多少？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="北京和杭州接下来几天天气如何？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

采用 stream 模式的 updates 模式，可以看到工具调用的具体步骤：

In [ ]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="467,123的平方根？")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()

工作流程：
![image-20260416141226554](https://cdn.jsdelivr.net/gh/XVSHIFU/Picture-bed@img/image-20260416141226554.png)

# 预定义工具

LangChain 提供了很多预定义好的工具：https://docs.langchain.com/oss/python/integrations/tools

例如：模型本身只能根据本身的训练数据回答问题，无法获取实时信息。但如果我们给它提供了web搜索的工具，那么你的Agent就如同具备了实时web搜索的能力，回答会更加准确

有一个专门用于给Agent提供Web搜索的工具，叫做Tavily，官网如下：

https://www.tavily.com/certification

LangChain已经集成了Tavily工具，我们可以直接使用：

https://docs.langchain.com/oss/python/integrations/tools/tavily_search


## Tavily工具使用
### 注册账号

获取 API_KEY

![image-20260416141727246](https://cdn.jsdelivr.net/gh/XVSHIFU/Picture-bed@img/image-20260416141727246.png)



### 配置环境变量

TAVILY_API_KEY=

![image-20260416142127299](https://cdn.jsdelivr.net/gh/XVSHIFU/Picture-bed@img/image-20260416142127299.png)

### 安装依赖
`uv add langchain-tavily`

### 使用工具

http://docs.langchain.com/oss/python/integrations/tools/tavily_search

https://docs.tavily.com/documentation/api-reference/endpoint/search

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_tavily import TavilySearch

# 初始化工具，并设置参数，具体参数设置参考官方文档
tavily_tool = TavilySearch(
    max_results = 5,
    topic = "general",
)

In [ ]:
# 调用工具
tavily_tool.invoke("杭州今天天气如何？")

### 结合智能体


In [ ]:
agent = create_agent(
    model="deepseek-chat",
    tools=[tavily_tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题"
)

# 调用工具
for chunk in agent.stream(
        {"messages": [HumanMessage(content="北京接下来5天天气如何?")]},
        stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()

### 优化

注意，LangChain提供的TavilySearch工具描述非常复杂，参数也很多。会有额外的网络消耗。如果我们仅仅是需要query参数，建议自定义工具。

In [ ]:
from langchain.tools import tool

# 使用 tavily 作为 web 搜索工具
tavily = TavilySearch(
    max_results = 5,
    topic = "general",
)

@tool
def web_search(query: str):
    """使用Tavily进行Web搜索，并返回标准化的来源结构。"""
    raw = tavily.invoke(query)
    items = raw.get("results", []) if isinstance(raw, dict) else []
    normalized = []
    for idx, item in enumerate(items, start=1):
        if not isinstance(item, dict):
            continue
        normalized.append(
            {
                "source_id": f"S{idx}",
                "title": item.get("title", ""),
                "url": item.get("url", ""),
                "snippet": item.get("content", "")[:220],
            }
        )

    return {
        "query": query,
        "results": normalized,
    }

默认情况下，AI回答的结果不包含信息来源，我们可以自定义结构化输出，让AI在回答的同时把信息来源也返回来

In [ ]:
import json
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Agent 回答内容引用的网页信息
class Reference(BaseModel):
    source_id: str = Field(description="Source id like S1, S2")
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")
    snippet: str = Field(description="A short snippet copied from the source")

# Agent 回答内容
class AnswerInfo(BaseModel):
    answer: str = Field(description="Final answer in Chinese. Use citations like [S1] in key statements")
    key_points: list[str] = Field(default_factory=list, description="3-5 bullet key points")
    references: list[Reference] = Field(default_factory=list, description="Cited web sources")

# 创建智能体，使用预定义工具 tavily_search
agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt=(
        "你是一个检索增强助手。请先调用web_search，再基于检索结果作答。"
        "输出必须符合AnswerInfo结构；关键事实要在answer中加来源标记，如[S1]。"
        "references字段只保留实际引用到的来源。"
    ),
    response_format=AnswerInfo
)

# 调用智能体
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

# 获取结构化输出
print(json.dumps(response["structured_response"].model_dump(), ensure_ascii=False, indent=2))
